# Part 1 — Ingest & Clean
**Input** : `Motor_Vehicle_Collisions_-_Crashes_20260411.csv`  
**Output** : `crash_manhattan_alcohol_clean.csv`

Steps:
1. Load raw NYC collision CSV
2. Filter Manhattan only
3. Filter alcohol-involved crashes
4. Report + geocode the 73 rows missing lat/lon
5. Save clean file

In [5]:
import pandas as pd

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(
    "Motor_Vehicle_Collisions_-_Crashes_20260411.csv",
    dtype=str,
    low_memory=False
)

# Normalize column names
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# ── Filter 1: Manhattan only ──────────────────────────────────────────────────
df_manhattan = df[df["borough"].str.strip().str.upper() == "MANHATTAN"].copy()
print(f"Total rows:         {len(df):,}")
print(f"Manhattan rows:     {len(df_manhattan):,}")

# ── Filter 2: Alcohol involvement in ANY of the 5 contributing factor cols ────
contrib_cols = [
    "contributing_factor_vehicle_1",
    "contributing_factor_vehicle_2",
    "contributing_factor_vehicle_3",
    "contributing_factor_vehicle_4",
    "contributing_factor_vehicle_5",
]

alcohol_mask = df_manhattan[contrib_cols].apply(
    lambda col: col.str.strip().str.lower() == "alcohol involvement"
).any(axis=1)

df_alcohol = df_manhattan[alcohol_mask].copy()
print(f"Alcohol-involved:   {len(df_alcohol):,}")

Total rows:         2,254,014
Manhattan rows:     346,125
Alcohol-involved:   2,951


In [6]:
# ── Missing lat/lon report ────────────────────────────────────────────────────
df_alcohol["latitude"]  = pd.to_numeric(df_alcohol["latitude"],  errors="coerce")
df_alcohol["longitude"] = pd.to_numeric(df_alcohol["longitude"], errors="coerce")

missing_lat  = df_alcohol["latitude"].isna().sum()
missing_lon  = df_alcohol["longitude"].isna().sum()
missing_both = df_alcohol[df_alcohol["latitude"].isna() & df_alcohol["longitude"].isna()]
total        = len(df_alcohol)

print(f"\n── Missing lat/lon report ──────────────────────────────")
print(f"Missing latitude:       {missing_lat:,}  ({missing_lat/total*100:.1f}%)")
print(f"Missing longitude:      {missing_lon:,}  ({missing_lon/total*100:.1f}%)")
print(f"Missing both:           {len(missing_both):,}  ({len(missing_both)/total*100:.1f}%)")

print("\n── Sample of rows missing lat/lon ──────────────────────")
print(missing_both[[
    "crash_date", "crash_time", "on_street_name",
    "cross_street_name", "off_street_name", "zip_code", "borough"
]].head(15).to_string(index=False))

# ── Do missing rows have street names we could geocode? ──────────────────────
has_on_street = missing_both["on_street_name"].notna().sum()
has_cross     = missing_both["cross_street_name"].notna().sum()
has_off       = missing_both["off_street_name"].notna().sum()

print(f"\nOf the {len(missing_both):,} missing-coord rows:")
print(f"  Have on_street_name:    {has_on_street:,}")
print(f"  Have cross_street_name: {has_cross:,}")
print(f"  Have off_street_name:   {has_off:,}")
print(f"  Have none of the above: {(missing_both[['on_street_name','cross_street_name','off_street_name']].isna().all(axis=1)).sum():,}")


── Missing lat/lon report ──────────────────────────────
Missing latitude:       73  (2.5%)
Missing longitude:      73  (2.5%)
Missing both:           73  (2.5%)

── Sample of rows missing lat/lon ──────────────────────
crash_date crash_time               on_street_name cross_street_name                 off_street_name zip_code   borough
03/26/2022       5:43                 GRAND STREET      essex street                             NaN    10002 MANHATTAN
09/22/2021      16:02              WEST 145 STREET          BROADWAY                             NaN    10031 MANHATTAN
07/24/2021       2:36 FREDERICK DOUGLASS BOULEVARD   WEST 142 STREET                             NaN    10030 MANHATTAN
08/01/2021       1:52                          NaN               NaN       805       RIVERSIDE DRIVE    10032 MANHATTAN
10/18/2021      21:39         SAINT NICHOLAS PLACE   WEST 153 STREET                             NaN    10031 MANHATTAN
02/12/2022       3:20       AVENUE OF THE AMERICAS    WEST 

In [7]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="dont_crash_or_burn_project", timeout=10)
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.5,
    max_retries=3,
    error_wait_seconds=5.0
)

def build_address(row):
    """
    Priority 1: intersection (on_street + cross_street) — most reliable
    Priority 2: off_street_name (already a full address e.g. '805 RIVERSIDE DRIVE')
    """
    on    = str(row["on_street_name"]).strip()    if pd.notna(row["on_street_name"])    else ""
    cross = str(row["cross_street_name"]).strip() if pd.notna(row["cross_street_name"]) else ""
    off   = str(row["off_street_name"]).strip()   if pd.notna(row["off_street_name"])   else ""

    if on and cross:
        return f"{on} & {cross}, Manhattan, New York, NY"
    elif off:
        return f"{off}, Manhattan, New York, NY"
    return None

def geocode_row(row):
    addr = build_address(row)
    if not addr:
        return None, None
    try:
        loc = geocode(addr)
        if loc:
            return loc.latitude, loc.longitude
    except Exception:
        pass
    return None, None

# ── Only run on the rows missing coordinates ──────────────────────────────────
missing_idx = df_alcohol[df_alcohol["latitude"].isna()].index

print(f"Geocoding {len(missing_idx)} rows — ~{len(missing_idx)} seconds at 1 req/1.5s...")

for idx in missing_idx:
    lat, lon = geocode_row(df_alcohol.loc[idx])
    df_alcohol.at[idx, "latitude"]  = lat
    df_alcohol.at[idx, "longitude"] = lon

still_missing = df_alcohol["latitude"].isna().sum()
recovered     = len(missing_idx) - still_missing

print(f"\nRecovered:     {recovered} / {len(missing_idx)}")
print(f"Still missing: {still_missing}")

Geocoding 73 rows — ~73 seconds at 1 req/1.5s...

Recovered:     29 / 73
Still missing: 44


In [8]:
# ── Save clean file ───────────────────────────────────────────────────────────
df_clean = df_alcohol.dropna(subset=["latitude", "longitude"]).copy()
df_clean.to_csv("crash_manhattan_alcohol_clean.csv", index=False)
print(f"Saved {len(df_clean):,} rows → crash_manhattan_alcohol_clean.csv")
df_clean.head()

Saved 2,907 rows → crash_manhattan_alcohol_clean.csv


,crash_date,crash_time,borough,zip_code,latitude,longitude,location,on_street_name,cross_street_name,off_street_name,...,contributing_factor_vehicle_2,contributing_factor_vehicle_3,contributing_factor_vehicle_4,contributing_factor_vehicle_5,collision_id,vehicle_type_code_1,vehicle_type_code_2,vehicle_type_code_3,vehicle_type_code_4,vehicle_type_code_5
1309,09/11/2021,22:05,MANHATTAN,10022,40.761610,-73.970760,"(40.76161, -73.97076)",PARK AVENUE,EAST 57 STREET,NaN,...,Unspecified,NaN,NaN,NaN,4455920,Sedan,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN
1866,03/26/2022,5:43,MANHATTAN,10002,40.716456,-73.988721,NaN,GRAND STREET,essex street,NaN,...,NaN,NaN,NaN,NaN,4514227,Sedan,NaN,NaN,NaN,NaN
1900,03/26/2022,10:17,MANHATTAN,10035,40.801140,-73.937690,"(40.80114, -73.93769)",EAST 121 STREET,3 AVENUE,NaN,...,Unspecified,NaN,NaN,NaN,4514158,Sedan,Sedan,NaN,NaN,NaN
1908,03/26/2022,22:00,MANHATTAN,10002,40.718174,-73.987040,"(40.718174, -73.98704)",NaN,NaN,138 DELANCEY STREET,...,Unspecified,Unspecified,NaN,NaN,4513778,Sedan,Sedan,Taxi,NaN,NaN
2701,04/13/2021,10:00,MANHATTAN,10010,40.737804,-73.982760,"(40.737804, -73.98276)",NaN,NaN,235 EAST 22 STREET,...,NaN,NaN,NaN,NaN,4408078,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN
